# Session 3: MCP 서버 연동을 통한 Agent 기능 확장

### Index
1. MCP(Model Context Protocol) 개념
2. MCP 서버 준비 (SQLite)
3. Strands Agent + MCP 연동
4. 자연어로 데이터 질의

---
## 1. MCP(Model Context Protocol)란?

MCP는 AI Agent가 **외부 도구/데이터 소스와 통신하기 위한 표준 프로토콜**입니다.

### Session 2와의 차이

| | Session 2 (직접 Tool) | Session 3 (MCP) |
|---|---|---|
| Tool 정의 | Python 함수로 직접 작성 | MCP 서버가 제공 |
| 데이터 접근 | 하드코딩 / 로컬 변수 | 외부 리소스 (DB, API, 파일 등) |
| 재사용성 | 해당 Agent에서만 사용 | 어떤 Agent든 연결 가능 |
| 확장성 | Tool 추가 시 코드 수정 | 서버만 바꾸면 됨 |

### MCP 구조

```
Agent (Client)  <- MCP Protocol ->  MCP Server (SQLite, API, 파일 등)
                - Tool 목록 조회
                - Tool 호출/응답
```

- **MCP Client**: Agent 쪽. 서버에 어떤 Tool이 있는지 물어보고, 필요할 때 호출
- **MCP Server**: 실제 기능 제공. DB 조회, API 호출 등을 Tool로 노출
- **Transport**: 통신 방식 (stdio, HTTP 등)

> 💡 MCP 서버는 이미 다양한 오픈소스가 있어서, 직접 만들지 않아도 바로 연결할 수 있습니다.  
> 오늘은 `mcp-server-sqlite`를 사용합니다.

---
## 2. MCP 서버 준비 (SQLite)

> SQLite MCP 서버를 사용하면 Agent가 **자연어로 DB를 질의**할 수 있습니다.  
> 먼저 샘플 데이터베이스를 만들어보겠습니다.

In [ ]:
# ! pip install strands-agents mcp

### 샘플 데이터베이스 생성

> 간단한 쇼핑몰 데이터를 만듭니다: 상품(products), 주문(orders)

In [ ]:
import sqlite3
import os

# DB 파일 경로
DB_PATH = "data/shop.db"

# 기존 파일 삭제 후 새로 생성
if os.path.exists(DB_PATH):
  os.remove(DB_PATH)

conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

# 상품 테이블
cursor.execute("""
CREATE TABLE products (
  id INTEGER PRIMARY KEY,
  name TEXT NOT NULL,
  category TEXT NOT NULL,
  price INTEGER NOT NULL,
  stock INTEGER NOT NULL
)
""")

# 주문 테이블
cursor.execute("""
CREATE TABLE orders (
  id INTEGER PRIMARY KEY,
  product_id INTEGER,
  quantity INTEGER NOT NULL,
  order_date TEXT NOT NULL,
  customer_name TEXT NOT NULL,
  FOREIGN KEY (product_id) REFERENCES products(id)
)
""")

# 샘플 상품 데이터
products = [
  (1, "맥북 프로 14", "전자기기", 2800000, 15),
  (2, "에어팟 프로", "전자기기", 359000, 50),
  (3, "파이썬 코딩의 기술", "도서", 32000, 100),
  (4, "AWS 교과서", "도서", 38000, 80),
  (5, "기계식 키보드", "전자기기", 150000, 30),
  (6, "모니터 암", "사무용품", 45000, 60),
  (7, "스탠딩 데스크", "사무용품", 580000, 10),
  (8, "노이즈캔슬링 헤드폰", "전자기기", 420000, 25),
]
cursor.executemany("INSERT INTO products VALUES (?, ?, ?, ?, ?)", products)

# 샘플 주문 데이터
orders = [
  (1, 1, 1, "2026-05-01", "김철수"),
  (2, 2, 2, "2026-05-01", "이영희"),
  (3, 3, 3, "2026-05-02", "박민수"),
  (4, 4, 1, "2026-05-02", "김철수"),
  (5, 1, 1, "2026-05-03", "정다은"),
  (6, 5, 2, "2026-05-03", "이영희"),
  (7, 2, 1, "2026-05-04", "최준호"),
  (8, 7, 1, "2026-05-04", "박민수"),
  (9, 6, 3, "2026-05-05", "김철수"),
  (10, 8, 1, "2026-05-05", "정다은"),
]
cursor.executemany("INSERT INTO orders VALUES (?, ?, ?, ?, ?)", orders)

conn.commit()
conn.close()

print(f"✅ 데이터베이스 생성 완료: {DB_PATH}")
print(f"   - products: {len(products)}개 상품")
print(f"   - orders: {len(orders)}개 주문")

### 데이터 확인

In [ ]:
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

print("=== 상품 목록 ===")
for row in cursor.execute("SELECT * FROM products"):
  print(row)

print("\n=== 최근 주문 5건 ===")
for row in cursor.execute("SELECT o.id, p.name, o.quantity, o.order_date, o.customer_name FROM orders o JOIN products p ON o.product_id = p.id ORDER BY o.order_date DESC LIMIT 5"):
  print(row)

conn.close()

---
## 3. Strands Agent + MCP 연동

> `mcp-server-sqlite`를 MCP Client로 연결하고, Agent에 붙입니다.  
> Agent는 MCP 서버가 제공하는 Tool 목록을 자동으로 인식합니다.

### mcp-server-sqlite가 제공하는 Tool들

| 분류 | Tool | 설명 |
|------|------|------|
| Query | `read_query` | SELECT 쿼리 실행 (읽기 전용) |
| Query | `write_query` | INSERT/UPDATE/DELETE 실행 |
| Query | `create_table` | 테이블 생성 (CREATE TABLE) |
| Schema | `list_tables` | 데이터베이스의 모든 테이블 목록 조회 |
| Schema | `describe_table` | 특정 테이블의 컬럼 정보 조회 |
| Analysis | `append_insight` | 분석 중 발견한 비즈니스 인사이트를 메모에 추가 |

In [ ]:
from mcp import stdio_client, StdioServerParameters
from strands import Agent
from strands.tools.mcp import MCPClient
from strands.models import BedrockModel

In [ ]:
model = BedrockModel(
  model_id="us.amazon.nova-2-lite-v1:0",
  region_name="us-east-1"
)

In [ ]:
# MCP Client 생성 — SQLite MCP 서버에 연결
sqlite_mcp = MCPClient(lambda: stdio_client(
  StdioServerParameters(
      command="uvx",
      args=["mcp-server-sqlite", "--db-path", "./data/shop.db"]
  )
))

In [ ]:
# MCP Client를 context manager로 사용하여 Agent 생성

agent = Agent(
    model=model,
    tools=[sqlite_mcp],
    system_prompt="""너는 쇼핑몰 데이터 분석가야. SQLite 데이터베이스에 연결되어 있어. 사용자의 질문에 SQL 쿼리를 실행해서 정확한 데이터를 기반으로 답변해. 한국어로 답변해."""
)

# MCP 서버가 제공하는 Tool 목록 확인
print("📋 사용 가능한 Tool 목록:")
print("=" * 50)
for tool_name, tool in agent.tool_registry.registry.items():
    desc = tool.tool_spec.get('description', '') if hasattr(tool, 'tool_spec') else ''
    print(f"  - {tool_name}: {desc[:60]}")

---
## 4. 자연어로 데이터 질의

> 이제 Agent에게 자연어로 질문하면, Agent가 알아서:  
> 1. 테이블 구조를 파악하고  
> 2. 적절한 SQL을 생성하고  
> 3. 실행 결과를 해석해서 답변합니다.

In [ ]:
system_prompt="""너는 쇼핑몰 데이터 분석가야. SQLite 데이터베이스에 연결되어 있어. 사용자의 질문에 SQL 쿼리를 실행해서 정확한 데이터를 기반으로 답변해. 한국어로 답변해."""

In [ ]:
# 기본 조회
agent = Agent(model=model, tools=[sqlite_mcp], system_prompt=system_prompt)

response = agent("어떤 테이블이 있고, 각 테이블에 어떤 컬럼이 있는지 알려줘")

In [ ]:
# 집계 질의
agent = Agent(model=model, tools=[sqlite_mcp], system_prompt=system_prompt)

response = agent("카테고리별 상품 수와 평균 가격을 알려줘")

In [ ]:
# JOIN 질의
agent = Agent(model=model, tools=[sqlite_mcp], system_prompt=system_prompt)

response = agent("가장 많이 주문한 고객은 누구야? 총 주문 금액도 알려줘")

In [ ]:
# 분석 질의
agent = Agent(model=model, tools=[sqlite_mcp], system_prompt=system_prompt)

response = agent("5월 3일 이후 주문 중에서 전자기기 카테고리 매출 합계는?")

In [ ]:
# 인사이트 요청
agent = Agent(model=model, tools=[sqlite_mcp], system_prompt=system_prompt)

response = agent("재고가 20개 이하인 상품 중에서 주문이 많은 상품은? 재입고 우선순위를 추천해줘")